# 144 — Proyecto: agente que actúa con límites

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Un agente que actúa (robot o computer use) no se vuelve desplegable por su tasa
de éxito, sino por sus **límites**. El proyecto integra cuatro capas de defensa:

- **Permisos (allowlist/denylist)**: cada acción se clasifica *antes* de
  ejecutarse. Allowlist = permitido (leer pantalla, mover cursor); denylist =
  prohibido (borrado definitivo, mover dinero); lo no clasificado es intermedio
  y **escala a un humano**. Principio de mínimo privilegio.
- **Sandbox**: barrera externa (contenedor, VM, límites de fuerza/velocidad de
  ISO/TS 15066) que actúa aunque el orquestador falle. No es lo mismo que un
  permiso: el permiso es una decisión de software; el sandbox, una barrera.
- **Aprobación humana**: se calibra por *reversibilidad × costo del error*.
  Escalarlo todo produce fatiga de aprobación y convierte la capa en teatro.
- **Auditoría**: toda acción, bloqueo y aprobación queda registrada; sin traza
  no hay evaluación de seguridad.

```text
proponer(accion) → clasificar:
    denylist   → rechazar y registrar
    allowlist  → ejecutar en sandbox y registrar
    intermedia → aprobación humana → (sí: ejecutar / no: replanificar)
```

La clasificación es determinista e independiente del modelo que propone: el
planificador puede equivocarse; la compuerta no depende de que no lo haga.


## 🧮 Ejemplo de referencia

Tarea: «organizar la carpeta de descargas». Traza con compuertas:

```text
1 listar(descargas/)              allowlist   → ejecuta
2 mover(a.pdf → docs/)            allowlist   → ejecuta
3 eliminar_definitivo(tmp.zip)    denylist    → RECHAZA  ⚠️
4 enviar_a_papelera(tmp.zip)      intermedia  → humano aprueba → ejecuta
5 vaciar_papelera()               denylist    → RECHAZA  ⚠️
```

Métricas: tarea completada; 3 acciones ejecutadas; **2 bloqueos de denylist**;
1 escalada aprobada. Lectura honesta: el episodio fue «exitoso», pero el
planificador propuso dos acciones prohibidas — hay que corregir el
planificador, no relajar la denylist.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("capstone", seed=144)
show(result)


## Reflexión

1. Si la denylist bloqueó 2 acciones en un episodio exitoso, ¿por qué el agente
   NO puede considerarse seguro, y qué componente hay que corregir primero?
2. ¿Qué diferencia operativa hay entre un permiso denegado por el orquestador y
   un límite impuesto por el sandbox, y por qué la defensa en profundidad exige
   ambos?
3. Si escalar todas las acciones intermedias a un humano produce fatiga de
   aprobación, ¿con qué dos variables calibrarías el umbral de escalada y qué
   métrica te avisaría de que el humano aprueba por hábito?
